# 简介

Search() 方法支持两种检索方式，在指定 namespace_prefix 下进行检索：
- 按 `filter` 做结构化过滤，关键词搜索。
- 按 `query` 做语义相似度检索，需要将输入转换为向量。向量搜索。

# 索引配置

这里结合自己的DualStruct项目理解，其中有一个概念是IndexProfile和这里的IndexConfig类似，配置本身用于描述一段向量嵌入过程所使用的参数。

```python
class IndexConfig(TypedDict, total=False):
    dims: int
    embed: Embeddings | EmbeddingsFunc | AEmbeddingsFunc | str
    fields: list[str] | None
```

### 参数说明
- **embed**：将输入文本转换为向量的嵌入函数，可以是自定义函数，也可以是嵌入模型对象，本例传递的是自定义嵌入函数。
- **dims**：输出向量维度
- **fields**：用于计算向量的属性列表，这里的属性都是指value中的key，value是一个JSON‑like字典，可取值如下
  - `["$"]`：将value作为整体嵌入
  - `["fields1", "fields2"]`：单独指定某个一级字段
  - `["parent.child"]`：从内部的嵌套JSON对象中获取子字段的值
  - `["array[*].field"]`：从JSON数组的每个JSON对象中获取子字段的值

> 注意：上述四种形式可以同时出现，fields列表的每个元素都会生成一个嵌入向量。

# 示例1-自定义嵌入函数

In [1]:
from langgraph.store.memory import InMemoryStore

# 自定义嵌入函数
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]

index_config = {
    "embed": embed,
    "dims": 6,
    "fields": ["$", "course"]
}

store = InMemoryStore(
    index = index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)


In [3]:
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


# 示例2-使用嵌入模型

In [14]:
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

import os
load_dotenv(override=True)

model_name = os.getenv("EMBEDDING_MODEL")

embedding_model = init_embeddings(
    model=f"openai:{model_name}",
    api_key=os.getenv("EMBEDDING_API_KEY"),
    base_url=os.getenv("EMBEDDING_API_URL"),
    dimensions=1024,
)

index_config = {
    "embed": embedding_model,
    "dims": 1024,
    "fields": ["$"]
}

store = InMemoryStore(
    index = index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

for item in store.search(("users", ), query="数电模电"):
    print(item)


NumPy not found in the current Python environment. The InMemoryStore will use a pure Python implementation for vector operations, which may significantly impact performance, especially for large datasets or frequent searches. For optimal speed and efficiency, consider installing NumPy: pip install numpy


Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T13:05:34.341453+00:00', updated_at='2026-09-01T13:05:34.341456+00:00', score=0.5057676926307514)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-01T13:05:34.491905+00:00', updated_at='2026-09-01T13:05:34.491907+00:00', score=0.46839559197753683)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-09-01T13:05:34.417552+00:00', updated_at='2026-09-01T13:05:34.417554+00:00', score=0.46611774759945)


# 示例3-理解 NumPy 警告与异常排序

## NumPy 只负责本地相似度计算

`InMemoryStore` 拿到远程嵌入模型返回的向量后，需要在本地计算查询向量与候选向量的余弦相似度。安装 NumPy 时使用矩阵运算；没有 NumPy 时退回普通 Python 循环，数学公式相同，主要差异是性能而不是排序结果。

本例通过 SiliconFlow API 在远程服务器上执行 Qwen3 Embedding 推理，所以不需要安装 PyTorch。只有改成本地加载 Hugging Face 模型时，才通常需要 PyTorch、Transformers 等依赖。

NumPy 在这里是可选的性能依赖。本示例数据量很小，保留纯 Python 实现即可，可以忽略该警告；本笔记不安装 NumPy。

## `fields=["$"]` 实际嵌入了什么

`$` 会把整个 value 序列化为一段 JSON 文本，再生成一个向量。因此课程、运动和食物会共同影响相似度，而不是只比较 `course`。

In [15]:
import importlib.util
import json

print("是否已安装 NumPy：", importlib.util.find_spec("numpy") is not None)
print("fields=['$'] 送入嵌入模型的文本：")
print(json.dumps(value1, sort_keys=True, ensure_ascii=False))
print("\nfields=['course'] 送入嵌入模型的文本：")
print(value1["course"])


是否已安装 NumPy： False
fields=['$'] 送入嵌入模型的文本：
{"course": "计算机组成原理", "food": "紫光园奶皮子酸奶", "sports": "跑步"}

fields=['course'] 送入嵌入模型的文本：
计算机组成原理


## 对照实验1：整个对象与指定字段

下面使用完全相同的三条数据建立两个 Store：

- `whole_store` 使用 `fields=["$"]`，课程、运动和食物共同形成一个向量。
- `course_store` 使用 `fields=["course"]`，只让课程参与语义检索。

Bob 和 Black 的课程完全相同。在 `course_store` 中，两者应获得相同分数；而在 `whole_store` 中，其他字段会让二者分数产生差异。

In [16]:
records = [
    (namespace1, key1, value1),
    (namespace2, key2, value2),
    (namespace3, key3, value3),
]

# 前面的 store 就是 fields=['$'] 的版本，直接复用，避免重复创建。
whole_store = store

course_store = InMemoryStore(
    index={
        "embed": embedding_model,
        "dims": 1024,
        "fields": ["course"],
    }
)

for namespace, key, value in records:
    course_store.put(namespace, key, value)

def show_scores(title: str, target_store: InMemoryStore, query: str) -> None:
    print(f"\n{title}｜query={query!r}")
    for item in target_store.search(("users",), query=query):
        user = item.namespace[1]
        print(f"{user:<5} course={item.value['course']:<12} score={item.score:.6f}")

show_scores("整个对象参与嵌入", whole_store, "数电模电")
show_scores("仅 course 字段参与嵌入", course_store, "数电模电")



整个对象参与嵌入｜query='数电模电'
Alice course=计算机组成原理      score=0.505768
Black course=数字电路与模拟电路    score=0.468396
Bob   course=数字电路与模拟电路    score=0.466118

仅 course 字段参与嵌入｜query='数电模电'
Bob   course=数字电路与模拟电路    score=0.464502
Black course=数字电路与模拟电路    score=0.464502
Alice course=计算机组成原理      score=0.370444


## 对照实验2：短缩写、完整名称与 Qwen instruction

`数电模电` 是较短的口语缩写，缺少明确的检索任务上下文。Qwen3 Embedding 是 instruction-aware 模型，官方建议在查询端使用下面的格式，文档端不需要添加：

```text
Instruct: <用一句话描述检索任务>
Query: <用户查询>
```

因此下面分别测试短缩写、完整课程名和带检索指令的短缩写。分数只能在同一模型、同一索引和同一任务内相对比较，不能简单理解成“超过 0.5 就是高相似度”。

In [17]:
queries = {
    "短缩写": "数电模电",
    "完整名称": "数字电路与模拟电路",
    "带检索指令": (
        "Instruct: Given a course-name query, retrieve records with "
        "semantically equivalent course names\n"
        "Query: 数电模电"
    ),
}

for label, query in queries.items():
    show_scores(label, course_store, query)



短缩写｜query='数电模电'
Bob   course=数字电路与模拟电路    score=0.464502
Black course=数字电路与模拟电路    score=0.464502
Alice course=计算机组成原理      score=0.370444

完整名称｜query='数字电路与模拟电路'
Bob   course=数字电路与模拟电路    score=1.000000
Black course=数字电路与模拟电路    score=1.000000
Alice course=计算机组成原理      score=0.336700

带检索指令｜query='Instruct: Given a course-name query, retrieve records with semantically equivalent course names\nQuery: 数电模电'
Bob   course=数字电路与模拟电路    score=0.517080
Black course=数字电路与模拟电路    score=0.517080
Alice course=计算机组成原理      score=0.405071


# 产品实践：`$` 与指定字段如何选择

正常产品中，**指定语义字段或专门构造 `search_text` 更常见**；`fields=["$"]` 更适合快速原型，以及字段少、语义一致、每个字段都应该影响召回的记忆对象。

| 场景 | 推荐方式 | 原因 |
| --- | --- | --- |
| 只按课程、标题、正文等明确内容搜索 | `fields=["course"]` 等指定字段 | 避免无关字段稀释语义 |
| 用户可能搜索任意一种偏好 | `fields=["course", "sports", "food"]` | 每个字段单独生成向量，`InMemoryStore` 取该条记录各字段中的最高相似度 |
| 字段很多且需要控制语义权重 | 额外构造 `search_text` | 可以加入字段标签、摘要、同义词和业务上下文 |
| 时间、状态、用户 ID、权限、类别 | 不做 embedding，使用 `filter` | 这些是精确约束，不适合交给语义相似度判断 |
| 字段少且整条对象本身就是一段完整记忆 | `fields=["$"]` | 简单直接，所有字段共同表达一个主题 |

典型产品通常采用混合检索思路：先用 namespace、权限、状态、类别等结构化条件缩小范围，再对 `title`、`content`、`summary` 或 `search_text` 做向量检索。`$` 是方便的默认值，但不应无条件用于包含大量异质字段的业务对象。